# Prompt-Based 情感分类（BERT + ChnSentiCorp）

## 项目概览
- **模型**：BERT-Base-Chinese (102M 参数)
- **任务**：情感分类 (Sentiment Analysis)，正面/负面二分类
- **数据集**：ChnSentiCorp（中文情感数据集）
- **方法**：Prompt-Based（提示学习），利用 MLM 头进行分类
- **评估指标**：Accuracy、Precision、Recall、F1

## Prompt-Based 分类 vs 传统分类
| 对比项 | 传统分类 | Prompt-Based（本项目） |
|--------|---------|------------------------|
| 输入 | 原始文本 | 模板：`总体上来说很[MASK]。{评论}` |
| 输出头 | 线性分类头 | MLM 头（预测 [MASK] 的词） |
| 标签映射 | 0/1 → softmax | 好（正面）/ 差（负面）→ verbalizer |
| 优点 | 简单直接 | 更贴近预训练目标，小样本效果好 |

## Prompt 设计
```
输入评论: "这家酒店的服务很好，值得推荐"
  ↓
Prompt:  "总体上来说很[MASK]。这家酒店的服务很好，值得推荐"
  ↓
模型预测 [MASK] 的词：
  P(好) → 正面情感
  P(差) → 负面情感
```

## 运行说明
- 使用快捷键 `Shift + Enter` 逐个运行 Cell
- 或点击菜单 `Cell → Run All` 一次性运行全部
- 整个流程耗时约 30-60 分钟（GPU）

## 第一步：环境检查和库导入

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig,
    AutoTokenizer,
    BertPreTrainedModel,
    BertModel,
    AdamW,
    get_scheduler
)
from transformers.activations import ACT2FN

from sklearn.metrics import classification_report
from tqdm.auto import tqdm

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")
print(f"✓ PyTorch 版本: {torch.__version__}")
print(f"✓ GPU 可用: {torch.cuda.is_available()}")
print(f"✓ MPS 可用: {torch.backends.mps.is_available()}")

## 第二步：数据加载

数据格式（TSV，制表符分隔）：
```
这家酒店的服务很好，值得推荐\t1
房间很脏，服务态度恶劣\t0
```
标签：1 = 正面，0 = 负面

In [ ]:
DATA_DIR   = '../../data/ChnSentiCorp/'
OUTPUT_DIR = './senti_model_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, 'train.txt')
DEV_FILE   = os.path.join(DATA_DIR, 'dev.txt')
TEST_FILE  = os.path.join(DATA_DIR, 'test.txt')

for name, path in [('训练数据', TRAIN_FILE), ('验证数据', DEV_FILE), ('测试数据', TEST_FILE)]:
    print(f"{name}: {path}")
    print(f"  存在: {os.path.exists(path)}")
    if os.path.exists(path):
        with open(path) as f:
            lines = f.readlines()
        print(f"  样本数: {len(lines)}")

print(f"\n输出目录: {OUTPUT_DIR}")

## 第三步：Prompt 设计和数据集类

**Prompt 模板设计原则**：
- 模板要自然，符合语言习惯
- [MASK] 位置要让模型容易预测出标签词
- 标签词（verbalizer）要和预训练语料中的语义一致

**本项目**：
- 模板：`总体上来说很[MASK]。{评论}`
- 正面标签词：`好`
- 负面标签词：`差`

In [ ]:
def get_prompt(comment):
    """生成 Prompt 模板，返回 prompt 文本和 [MASK] 的字符位置"""
    prompt = f'总体上来说很[MASK]。{comment}'
    return {
        'prompt':      prompt,
        'mask_offset': prompt.find('[MASK]')
    }


def get_verbalizer(tokenizer):
    """定义标签词（verbalizer）：正面→好，负面→差"""
    return {
        'pos': {'token': '好', 'id': tokenizer.convert_tokens_to_ids('好')},
        'neg': {'token': '差', 'id': tokenizer.convert_tokens_to_ids('差')}
    }


class ChnSentiCorp(Dataset):
    """中文情感分类数据集（TSV 格式）"""

    def __init__(self, data_file):
        self.data = self.load_data(data_file)

    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                items = line.strip().split('\t')
                if len(items) != 2:
                    continue
                prompt_data = get_prompt(items[0])
                Data[idx] = {
                    'comment':     items[0],
                    'prompt':      prompt_data['prompt'],
                    'mask_offset': prompt_data['mask_offset'],
                    'label':       items[1]
                }
        return Data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

print("✓ Prompt 函数和 ChnSentiCorp 数据集类定义成功")

## 第四步：加载分词器和数据

In [ ]:
# ============ 模型选择配置 ============
# 1. 'bert-base-chinese'             - Google BERT（中文，推荐）⭐⭐⭐ (默认)
# 2. 'hfl/chinese-roberta-wwm-ext'   - 哈工大 RoBERTa（效果更好）⭐⭐⭐
# 3. 'hfl/chinese-macbert-base'      - MacBERT（特别适合 MLM Prompt）⭐⭐⭐
#
MODEL_CHOICE = 'bert-base-chinese'  # ⬅️ 修改这里选择模型
model_name   = MODEL_CHOICE

print(f"✓ 模型选择: {model_name}")
print("\n加载分词器...")
tokenizer  = AutoTokenizer.from_pretrained(model_name)
verbalizer = get_verbalizer(tokenizer)
print("✓ 分词器加载成功")
print(f"  正面标签词: '{verbalizer['pos']['token']}' (id={verbalizer['pos']['id']})")
print(f"  负面标签词: '{verbalizer['neg']['token']}' (id={verbalizer['neg']['id']})")

print("\n加载数据集...")
train_dataset = ChnSentiCorp(TRAIN_FILE)
dev_dataset   = ChnSentiCorp(DEV_FILE)
test_dataset  = ChnSentiCorp(TEST_FILE)
print(f"✓ 训练集: {len(train_dataset)} 条")
print(f"✓ 验证集: {len(dev_dataset)} 条")
print(f"✓ 测试集: {len(test_dataset)} 条")

## 第五步：查看数据样本

In [ ]:
for i in range(3):
    sample = train_dataset[i]
    label_text = '正面 ✓' if sample['label'] == '1' else '负面 ✗'
    print(f"\n=== 样本 {i+1} ===")
    print(f"原始评论: {sample['comment'][:60]}{'...' if len(sample['comment']) > 60 else ''}")
    print(f"Prompt:   {sample['prompt'][:70]}...")
    print(f"MASK 位置: 字符 {sample['mask_offset']}")
    print(f"标签:     {label_text}")

# 标签分布
from collections import Counter
labels = [train_dataset[i]['label'] for i in range(len(train_dataset))]
counter = Counter(labels)
print("\n训练集标签分布:")
print(f"  正面 (1): {counter['1']} 条 ({100*counter['1']/len(labels):.1f}%)")
print(f"  负面 (0): {counter['0']} 条 ({100*counter['0']/len(labels):.1f}%)")

## 第六步：超参数配置

In [ ]:
class Args:
    max_length       = 512
    batch_size       = 4
    num_train_epochs = 3
    learning_rate    = 1e-5
    warmup_proportion= 0.1
    weight_decay     = 0.01
    adam_beta1       = 0.9
    adam_beta2       = 0.98
    adam_epsilon     = 1e-8
    seed             = 12
    output_dir       = OUTPUT_DIR

args = Args()
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

if torch.backends.mps.is_available():
    args.device = torch.device('mps')
elif torch.cuda.is_available():
    args.device = torch.device('cuda')
else:
    args.device = torch.device('cpu')

print("超参数配置:")
for k, v in vars(args).items():
    print(f"  {k}: {v}")

## 第七步：定义模型

**Prompt-Based 模型结构**：
```
输入: [CLS] 总 体 上 来 说 很 [MASK] 。 评 论 ... [SEP]
  ↓
BERT Encoder → [batch, seq_len, hidden_size]
  ↓
取 [MASK] 位置的向量 → [batch, hidden_size]
  ↓
MLM Head（BERT 预训练头）→ [batch, vocab_size]
  ↓
只取 label_word_id（好/差）的 logits → [batch, 2]
  ↓
argmax → 0（负面）或 1（正面）
```

In [ ]:
def batched_index_select(input_tensor, dim, index):
    """按 batch 维度选取指定位置的向量"""
    for i in range(1, len(input_tensor.shape)):
        if i != dim:
            index = index.unsqueeze(i)
    expanse = list(input_tensor.shape)
    expanse[0] = -1
    expanse[dim] = -1
    index = index.expand(expanse)
    return torch.gather(input_tensor, dim, index)


class BertPredictionHeadTransform(nn.Module):
    """MLM 头的 Transform 层（dense + activation + LayerNorm）"""
    def __init__(self, config):
        super().__init__()
        self.dense         = nn.Linear(config.hidden_size, config.hidden_size)
        self.transform_act = ACT2FN[config.hidden_act] if isinstance(config.hidden_act, str) else config.hidden_act
        self.LayerNorm     = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, hidden_states):
        return self.LayerNorm(self.transform_act(self.dense(hidden_states)))


class BertLMPredictionHead(nn.Module):
    """MLM 预测头：将隐状态映射到词汇表大小"""
    def __init__(self, config):
        super().__init__()
        self.transform = BertPredictionHeadTransform(config)
        self.decoder   = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.bias      = nn.Parameter(torch.zeros(config.vocab_size))
        self.decoder.bias = self.bias

    def forward(self, hidden_states):
        return self.decoder(self.transform(hidden_states))


class BertForPrompt(BertPreTrainedModel):
    """
    Prompt-Based BERT 情感分类模型

    核心：用 MLM 头预测 [MASK] 位置的词（好/差），
    只取这两个词的 logits 做二分类。
    """

    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.cls  = BertLMPredictionHead(config)
        self.post_init()

    def get_output_embeddings(self):
        return self.cls.decoder

    def set_output_embeddings(self, new_embeddings):
        self.cls.decoder = new_embeddings

    def forward(self, batch_inputs, batch_mask_idxs, label_word_id, labels=None):
        bert_output     = self.bert(**batch_inputs)
        sequence_output = bert_output.last_hidden_state  # [B, seq_len, H]

        # 取 [MASK] 位置的向量
        batch_mask_reps = batched_index_select(
            sequence_output, 1, batch_mask_idxs.unsqueeze(-1)
        ).squeeze(1)  # [B, H]

        # MLM 头：映射到词汇表，只保留 label_word_id（好/差）的分数
        pred_scores = self.cls(batch_mask_reps)[:, label_word_id]  # [B, 2]

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(pred_scores, labels)

        return loss, pred_scores

print("✓ BertForPrompt 类定义成功")

## 第八步：创建 DataLoader

**关键**：在 collate 时需要找到每个样本中 `[MASK]` token 在 token 序列中的位置（`char_to_token`），以便后续取出该位置的隐状态做预测。

In [ ]:
def to_device(args, batch_data):
    new_batch_data = {}
    for k, v in batch_data.items():
        if k == 'batch_inputs':
            new_batch_data[k] = {k_: v_.to(args.device) for k_, v_ in v.items()}
        elif k == 'label_word_id':
            new_batch_data[k] = v  # list，不转 tensor
        else:
            new_batch_data[k] = torch.tensor(v).to(args.device)
    return new_batch_data


def get_dataLoader(args, dataset, tokenizer, verbalizer, shuffle=False):
    """创建 Prompt-Based 情感分类的 DataLoader"""
    pos_id = verbalizer['pos']['id']
    neg_id = verbalizer['neg']['id']

    def collote_fn(batch_samples):
        batch_sentences, batch_mask_idxs, batch_labels = [], [], []
        for sample in batch_samples:
            batch_sentences.append(sample['prompt'])
            # 找 [MASK] 在 token 序列中的位置
            encoding = tokenizer(sample['prompt'], truncation=True)
            mask_idx = encoding.char_to_token(sample['mask_offset'])
            assert mask_idx is not None, f"[MASK] 未找到，请检查 prompt 格式"
            batch_mask_idxs.append(mask_idx)
            batch_labels.append(int(sample['label']))

        batch_inputs = tokenizer(
            batch_sentences,
            max_length=args.max_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )

        return {
            'batch_inputs':    batch_inputs,
            'batch_mask_idxs': batch_mask_idxs,
            'label_word_id':   [neg_id, pos_id],  # 0=负面(差), 1=正面(好)
            'labels':          batch_labels
        }

    return DataLoader(dataset, batch_size=args.batch_size, shuffle=shuffle, collate_fn=collote_fn)


print("创建 DataLoader...")
train_dataloader = get_dataLoader(args, train_dataset, tokenizer, verbalizer, shuffle=True)
dev_dataloader   = get_dataLoader(args, dev_dataset,   tokenizer, verbalizer, shuffle=False)
test_dataloader  = get_dataLoader(args, test_dataset,  tokenizer, verbalizer, shuffle=False)

print(f"✓ 训练集批次数: {len(train_dataloader)}")
print(f"✓ 验证集批次数: {len(dev_dataloader)}")
print(f"✓ 测试集批次数: {len(test_dataloader)}")

## 第九步：加载模型

In [ ]:
print(f"加载 BERT Prompt 模型: {model_name}...")
config = AutoConfig.from_pretrained(model_name)
model  = BertForPrompt.from_pretrained(model_name, config=config)
model  = model.to(args.device)

print(f"✓ 模型加载成功")
print(f"✓ 模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"✓ 模型所在设备: {next(model.parameters()).device}")

## 第十步：设置优化器和学习率调度器

In [ ]:
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': args.weight_decay},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

t_total = len(train_dataloader) * args.num_train_epochs
args.warmup_steps = int(t_total * args.warmup_proportion)

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=args.learning_rate,
    betas=(args.adam_beta1, args.adam_beta2),
    eps=args.adam_epsilon
)
lr_scheduler = get_scheduler(
    'linear', optimizer,
    num_warmup_steps=args.warmup_steps,
    num_training_steps=t_total
)

print(f"优化器: AdamW，学习率: {args.learning_rate}")
print(f"调度器: Linear Warmup，总步数: {t_total}，预热步数: {args.warmup_steps}")

## 第十一步：定义训练和评估函数

In [ ]:
def train_loop(args, dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    """训练一个 epoch"""
    model.train()
    finish_step_num = epoch * len(dataloader)
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1} Training')

    for step, batch_data in enumerate(progress_bar, start=1):
        batch_data = to_device(args, batch_data)
        outputs    = model(**batch_data)
        loss       = outputs[0]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'avg_loss': f'{total_loss / (finish_step_num + step):.4f}'})

    return total_loss


def test_loop(args, dataloader, model):
    """评估：收集预测和真实标签，用 sklearn 计算分类报告"""
    true_labels, predictions = [], []

    model.eval()
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc='Evaluating'):
            true_labels += batch_data['labels']
            batch_data   = to_device(args, batch_data)
            outputs      = model(**batch_data)
            logits       = outputs[1]
            predictions += logits.argmax(dim=-1).cpu().numpy().tolist()

    return classification_report(true_labels, predictions, output_dict=True)

print("✓ train_loop / test_loop 定义成功")

## 第十二步：开始训练 ⭐⭐⭐

⚠️ **训练时间**：约 30-60 分钟（GPU）

最佳模型按 **(macro F1 + weighted F1) / 2** 选择。

In [ ]:
training_history = {
    'train_loss':      [],
    'dev_accuracy':    [],
    'dev_macro_f1':    [],
    'dev_weighted_f1': []
}

best_score      = 0.0
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.bin')
total_loss      = 0.0

print("开始训练...\n")
for epoch in range(args.num_train_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{args.num_train_epochs}")
    print(f"{'='*60}")

    total_loss = train_loop(args, train_dataloader, model, optimizer, lr_scheduler, epoch, total_loss)
    avg_loss   = total_loss / ((epoch + 1) * len(train_dataloader))
    print(f"\n平均训练损失: {avg_loss:.4f}")
    training_history['train_loss'].append(avg_loss)

    metrics     = test_loop(args, dev_dataloader, model)
    accuracy    = metrics['accuracy']
    macro_f1    = metrics['macro avg']['f1-score']
    weighted_f1 = metrics['weighted avg']['f1-score']
    score       = (macro_f1 + weighted_f1) / 2

    print(f"\n验证集指标:")
    print(f"  Accuracy:    {100*accuracy:.2f}%")
    print(f"  macro F1:    {100*macro_f1:.2f}")
    print(f"  weighted F1: {100*weighted_f1:.2f}")
    print(f"  正面: P={100*metrics['1']['precision']:.2f}  R={100*metrics['1']['recall']:.2f}  F1={100*metrics['1']['f1-score']:.2f}")
    print(f"  负面: P={100*metrics['0']['precision']:.2f}  R={100*metrics['0']['recall']:.2f}  F1={100*metrics['0']['f1-score']:.2f}")

    training_history['dev_accuracy'].append(100 * accuracy)
    training_history['dev_macro_f1'].append(100 * macro_f1)
    training_history['dev_weighted_f1'].append(100 * weighted_f1)

    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), best_model_path)
        print(f"\n✓ 保存最佳模型 (score: {100*best_score:.2f})")

print("\n" + "="*60)
print("✅ 训练完成！")
print("="*60)

## 第十三步：绘制训练曲线

In [ ]:
epochs = range(1, args.num_train_epochs + 1)
fig, (ax_loss, ax_score) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('模型训练收敛曲线', fontsize=15, fontweight='bold')

ax_loss.plot(epochs, training_history['train_loss'], marker='o', linewidth=2, markersize=8)
ax_loss.set_xlabel('Epoch', fontsize=12)
ax_loss.set_ylabel('Loss', fontsize=12)
ax_loss.set_title('训练 Loss 曲线', fontsize=13, fontweight='bold')
ax_loss.set_xticks(epochs)
ax_loss.grid(True, alpha=0.3)

ax_score.plot(epochs, training_history['dev_accuracy'],    label='Accuracy', marker='o', linewidth=2)
ax_score.plot(epochs, training_history['dev_macro_f1'],    label='macro F1', marker='s', linewidth=2)
ax_score.plot(epochs, training_history['dev_weighted_f1'], label='weighted F1', marker='^', linewidth=2)
ax_score.set_xlabel('Epoch', fontsize=12)
ax_score.set_ylabel('分数 (%)', fontsize=12)
ax_score.set_title('验证集指标曲线', fontsize=13, fontweight='bold')
ax_score.set_xticks(epochs)
ax_score.legend(fontsize=11)
ax_score.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ 曲线已保存")

## 第十四步：加载最佳模型和预测函数

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=args.device))
model.eval()
print(f"✓ 加载最佳模型: {best_model_path}")


def predict_sentiment(comment, model, tokenizer, verbalizer, args):
    """
    对单条评论进行情感预测

    返回:
    {
      "label": 1,          # 1=正面, 0=负面
      "label_text": "正面",
      "confidence": 0.98,  # 预测置信度
      "prompt": "...",
      "mask_word": "好"    # 预测的标签词
    }
    """
    prompt_data = get_prompt(comment)
    prompt      = prompt_data['prompt']

    encoding = tokenizer(prompt, truncation=True)
    mask_idx = encoding.char_to_token(prompt_data['mask_offset'])

    inputs = tokenizer(
        prompt,
        max_length=args.max_length,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )

    batch_data = {
        'batch_inputs':    inputs,
        'batch_mask_idxs': [mask_idx],
        'label_word_id':   [verbalizer['neg']['id'], verbalizer['pos']['id']]
    }
    batch_data = to_device(args, batch_data)

    model.eval()
    with torch.no_grad():
        outputs = model(**batch_data)
        logits  = outputs[1]
        prob    = torch.softmax(logits, dim=-1)

    pred       = logits.argmax(dim=-1)[0].item()
    confidence = prob[0][pred].item()
    mask_word  = verbalizer['pos']['token'] if pred == 1 else verbalizer['neg']['token']

    return {
        'label':      pred,
        'label_text': '正面' if pred == 1 else '负面',
        'confidence': confidence,
        'prompt':     prompt,
        'mask_word':  mask_word
    }

print("✓ predict_sentiment 函数定义成功")

## 第十五步：测试集预测示例

In [ ]:
print("\n" + "="*80)
print("测试集预测示例（前 5 条）")
print("="*80)

for i in range(min(5, len(test_dataset))):
    sample = test_dataset[i]
    result = predict_sentiment(sample['comment'], model, tokenizer, verbalizer, args)

    true_label = '正面' if sample['label'] == '1' else '负面'
    match = '✓' if result['label_text'] == true_label else '✗'

    print(f"\n示例 {i+1}: {match}")
    print(f"评论:     {sample['comment'][:60]}{'...' if len(sample['comment']) > 60 else ''}")
    print(f"真实标签: {true_label}")
    print(f"预测结果: {result['label_text']}（[MASK]→'{result['mask_word']}'，置信度: {result['confidence']:.3f}）")
    print("-" * 80)

## 第十六步：自定义输入预测

In [ ]:
custom_comments = [
    "酒店位置很好，房间干净整洁，服务热情，非常满意，下次还会入住。",
    "房间太小了，隔音很差，价格贵但体验很差，不推荐。",
    "交通便利，早餐丰盛，整体性价比不错，值得推荐。",
    "服务态度极差，前台冷漠，房间有异味，完全不值这个价。",
]

print("\n" + "="*80)
print("自定义输入情感预测")
print("="*80)

for idx, comment in enumerate(custom_comments):
    result = predict_sentiment(comment, model, tokenizer, verbalizer, args)
    emoji  = '😊' if result['label_text'] == '正面' else '😢'
    print(f"\n示例 {idx+1}: {emoji}")
    print(f"评论: {comment}")
    print(f"情感: {result['label_text']}（[MASK]预测为'{result['mask_word']}'，置信度: {result['confidence']:.3f}）")

## 第十七步：测试集最终评估

In [ ]:
print("在测试集上评估...")
test_metrics = test_loop(args, test_dataloader, model)

print("\n测试集详细报告:")
print(f"  Accuracy:    {100*test_metrics['accuracy']:.2f}%")
print(f"  macro F1:    {100*test_metrics['macro avg']['f1-score']:.2f}")
print(f"  weighted F1: {100*test_metrics['weighted avg']['f1-score']:.2f}")
label_names = {0: '负面', 1: '正面'}
for k in ['0', '1']:
    m = test_metrics[k]
    print(f"  {label_names[int(k)]}: P={100*m['precision']:.2f}  R={100*m['recall']:.2f}  F1={100*m['f1-score']:.2f}")

## 第十八步：模型保存和总结

In [ ]:
model_save_path = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(model_save_path, exist_ok=True)
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ 模型已保存到: {model_save_path}")

with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w', encoding='utf-8') as f:
    json.dump(training_history, f, indent=2)

print("\n" + "="*80)
print("✅ 项目总结")
print("="*80)
print(f"""
【模型信息】
  架构: BERT-Base-Chinese + MLM Head（Prompt-Based）
  任务: 情感分类（正面/负面）
  数据集: ChnSentiCorp
  设备: {args.device}

【Prompt 设计】
  模板: 总体上来说很[MASK]。{{评论}}
  正面标签词: 好（verbalizer: pos）
  负面标签词: 差（verbalizer: neg）

【最终指标（测试集）】
  Accuracy:    {100*test_metrics['accuracy']:.2f}%
  macro F1:    {100*test_metrics['macro avg']['f1-score']:.2f}
  weighted F1: {100*test_metrics['weighted avg']['f1-score']:.2f}

【下一步】
  1. 尝试 MacBERT（hfl/chinese-macbert-base），MLM 预训练更适合 Prompt
  2. 使用 virtual label words（虚拟标签词）替代 base verbalizer
  3. 加载保存的模型:
     config    = AutoConfig.from_pretrained('{model_save_path}')
     tokenizer = AutoTokenizer.from_pretrained('{model_save_path}')
     model     = BertForPrompt.from_pretrained('{model_save_path}', config=config)
""")
print("="*80)